# Synthetic Image Degradation Analysis

Tests quality assessment methods against controlled synthetic degradations.

**Uses EXACT same methods as pairwise benchmark** (`configs/pairwise_benchmark.test.yaml`)

**Purpose**: Validate methods respond sensibly to known quality changes:
- Gaussian blur (tests sharpness detection)
- Exposure adjustment (tests exposure/histogram quality)  
- JPEG compression (tests artifact detection)

**Expected**: Quality scores decrease monotonically as degradation increases.

## Setup & Configuration

In [ ]:
# ============================================
# ENVIRONMENT VERIFICATION & SETUP
# ============================================
import sys
from pathlib import Path
from datetime import datetime

# Add project root to path (needed to import sim_bench modules)
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import utilities
from sim_bench.analysis.utils import get_project_root

# Get project root
PROJECT_ROOT = get_project_root()

# Verify Python environment
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Project root: {PROJECT_ROOT}")
print()

# Import fivecentplots (but we won't use it - keeping code for future use)
try:
    import fivecentplots as fcp
    print(f"✓ fivecentplots imported successfully (not used in current analysis)")
    print(f"  Location: {fcp.__file__}")
except ImportError as e:
    print(f"⚠ fivecentplots not available (not required for current analysis)")

# Import other required modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import cv2

# Import sim_bench modules
from sim_bench.image_processing import create_degradation_processor
from sim_bench.quality_assessment import create_quality_assessor

# Reset and set plotting style (fivecentplots may modify matplotlib defaults)
plt.rcdefaults()  # Reset to matplotlib defaults
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

# Set seaborn style last to ensure it's applied
sns.set_style('whitegrid')

print()
print("Environment verified ✓")


In [ ]:
# ============================================
# CONFIGURATION - EXACT COPY from pairwise_benchmark.test.yaml
# ============================================
# Test images from examples/images (relative to PROJECT_ROOT)
TEST_IMAGES = [
    "examples/images/portrait_01.jpg",
    "examples/images/portrait_02.jpg",
    "examples/images/landscape_01.jpg",
    "examples/images/landscape_02.jpg",
    "examples/images/macro_01.jpg",
]

# Resolve image paths relative to PROJECT_ROOT
TEST_IMAGES = [PROJECT_ROOT / path for path in TEST_IMAGES]

# Degradation levels
BLUR_SIGMAS = [0.5, 1.0, 2.0, 4.0, 8.0]
EXPOSURE_STOPS = [-3, -2, -1, 1, 2, 3]
JPEG_QUALITIES = [95, 80, 60, 40, 20, 10]

# Quality assessment methods - EXACT COPY from configs/pairwise_benchmark.test.yaml
METHODS_TO_TEST = [
    # Rule-based methods (5 methods)
    {'name': 'Sharpness', 'type': 'rule_based', 'weights': {'sharpness': 1.0, 'exposure': 0.0, 'colorfulness': 0.0, 'contrast': 0.0}},
    {'name': 'Exposure', 'type': 'rule_based', 'weights': {'sharpness': 0.0, 'exposure': 1.0, 'colorfulness': 0.0, 'contrast': 0.0}},
    {'name': 'Contrast', 'type': 'rule_based', 'weights': {'sharpness': 0.0, 'exposure': 0.0, 'colorfulness': 0.0, 'contrast': 1.0}},
    {'name': 'Colorfulness', 'type': 'rule_based', 'weights': {'sharpness': 0.0, 'exposure': 0.0, 'colorfulness': 1.0, 'contrast': 0.0}},
    {'name': 'Combined-RuleBased', 'type': 'rule_based', 'weights': {'sharpness': 0.4, 'exposure': 0.3, 'colorfulness': 0.2, 'contrast': 0.1}},

    # CLIP methods with attribute-specific prompts (7 methods)
    {'name': 'CLIP-Aesthetic-Overall', 'type': 'clip_aesthetic', 'model_name': 'ViT-B-32', 'device': 'cpu', 'aggregation_method': 'contrastive_only', 'custom_prompts': {'contrastive': ['a highly aesthetic, visually pleasing, beautiful photograph', 'an unattractive, poorly composed, ugly photograph'], 'positive': [], 'negative': []}},
    {'name': 'CLIP-Composition', 'type': 'clip_aesthetic', 'model_name': 'ViT-B-32', 'device': 'cpu', 'aggregation_method': 'contrastive_only', 'custom_prompts': {'contrastive': ['a well-composed photograph with excellent visual balance', 'a poorly-composed photograph with bad visual balance'], 'positive': [], 'negative': []}},
    {'name': 'CLIP-Subject-Placement', 'type': 'clip_aesthetic', 'model_name': 'ViT-B-32', 'device': 'cpu', 'aggregation_method': 'contrastive_only', 'custom_prompts': {'contrastive': ['a photo with the subject well placed in the frame', 'a photo with the subject not well placed in the frame'], 'positive': [], 'negative': []}},
    {'name': 'CLIP-Cropping', 'type': 'clip_aesthetic', 'model_name': 'ViT-B-32', 'device': 'cpu', 'aggregation_method': 'contrastive_only', 'custom_prompts': {'contrastive': ['a photo that is well cropped and shows the complete subject', 'a photo that is poorly cropped or cuts off the subject'], 'positive': [], 'negative': []}},
    {'name': 'CLIP-Sharpness', 'type': 'clip_aesthetic', 'model_name': 'ViT-B-32', 'device': 'cpu', 'aggregation_method': 'contrastive_only', 'custom_prompts': {'contrastive': ['a sharp, in-focus photograph with clear details', 'a blurry, out-of-focus photograph with unclear details'], 'positive': [], 'negative': []}},
    {'name': 'CLIP-Exposure', 'type': 'clip_aesthetic', 'model_name': 'ViT-B-32', 'device': 'cpu', 'aggregation_method': 'contrastive_only', 'custom_prompts': {'contrastive': ['a photo with good exposure and lighting', 'a photo with poor exposure, too dark or too bright'], 'positive': [], 'negative': []}},
    {'name': 'CLIP-Color', 'type': 'clip_aesthetic', 'model_name': 'ViT-B-32', 'device': 'cpu', 'aggregation_method': 'contrastive_only', 'custom_prompts': {'contrastive': ['a photo with vibrant, natural colors', 'a photo with dull, washed out colors'], 'positive': [], 'negative': []}},

    # Deep Learning Methods
    {'name': 'NIMA-MobileNet', 'type': 'nima', 'backbone': 'mobilenet_v2', 'device': 'cpu', 'batch_size': 8},
    {'name': 'NIMA-ResNet50', 'type': 'nima', 'backbone': 'resnet50', 'device': 'cpu', 'batch_size': 4},
    # {'name': 'MUSIQ', 'type': 'musiq', 'use_pyiqa': True, 'device': 'cpu', 'batch_size': 1},
    {'name': 'CLIP-Aesthetic-LAION', 'type': 'clip_aesthetic', 'variant': 'laion', 'model_name': 'ViT-B-32', 'device': 'cpu'},
]

# Device (can change to 'cuda' for GPU)
DEVICE = 'cpu'

# Output directory
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'degradation_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
print(f"  Test images: {len(TEST_IMAGES)}")
print(f"  Methods: {len(METHODS_TO_TEST)}")
print(f"  Device: {DEVICE}") 
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
df= pd.read_csv(r'D:\sim-bench\outputs\pairwise_benchmark_test\pairwise_20251120_231551\detailed_results.csv')
df['method'].to_list()

In [ ]:
df= pd.read_csv(r'D:\sim-bench\outputs\pairwise_benchmark_test\pairwise_20251120_231551\detailed_results.csv')
df['method'].to_list()

In [ ]:
df= pd.read_csv(r'D:\sim-bench\outputs\pairwise_benchmark_test\pairwise_20251120_231551\detailed_results.csv')
df['method'].to_list()

In [ ]:
df= pd.read_csv(r'D:\sim-bench\outputs\pairwise_benchmark_test\pairwise_20251120_231551\detailed_results.csv')
df['method'].to_list()

## Generate Degradations

In [ ]:
if not TEST_IMAGES:
    print("⚠️ WARNING: No test images provided!")
    print("Please add image paths to TEST_IMAGES list in the configuration cell above.")
else:
    # Create timestamped output directory
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    degradation_dir = OUTPUT_DIR / f'degradations_{timestamp}'
    degradation_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating degraded variants in: {degradation_dir}")

    # Create degradation processor
    processor = create_degradation_processor(output_dir=degradation_dir)

    # Generate degradations for each test image
    all_degraded_paths = {}

    for img_path in tqdm(TEST_IMAGES, desc="Generating degradations"):
        img_path = Path(img_path)

        degraded_variants = processor.apply_degradation_suite(
            image_path=img_path,
            blur_sigmas=BLUR_SIGMAS,
            exposure_stops=EXPOSURE_STOPS,
            jpeg_qualities=JPEG_QUALITIES,
            output_base_dir=degradation_dir
        )

        all_degraded_paths[img_path.stem] = degraded_variants

    total_variants = sum(len(v) for v in all_degraded_paths.values())
    print(f"\n✓ Generated {total_variants} total image variants")

## Quality Assessment

In [ ]:
print("Assessing quality for all degraded variants...")
print(f"Methods: {[m['name'] for m in METHODS_TO_TEST]}")

# Store all results
results = []

# Assess each method
for method_config in tqdm(METHODS_TO_TEST, desc="Testing methods"):
    method_name = method_config['name']
    assessor = create_quality_assessor(method_config)

    # Assess all degraded variants
    for image_name, degraded_paths in all_degraded_paths.items():
        for degradation_name, img_path in tqdm(
            degraded_paths.items(),
            desc=f"{method_name} - {image_name}",
            leave=False
        ):
            score = assessor.assess_image(str(img_path))

            # Parse degradation type and level
            parts = degradation_name.split('_')
            degradation_type = parts[0]
            degradation_level = '_'.join(parts[1:]) if len(parts) > 1 else 'original'

            results.append({
                'image_name': image_name,
                'degradation_type': degradation_type,
                'degradation_level': degradation_level,
                'degradation_full': degradation_name,
                'method': method_name,
                'score': score,
                'file_path': str(img_path)
            })

# Create DataFrame
results_df = pd.DataFrame(results)

# Save results
results_csv = OUTPUT_DIR / f'assessment_results_{timestamp}.csv'
results_df.to_csv(results_csv, index=False)

print(f"\n✓ Completed {len(results)} quality assessments")
print(f"✓ Saved results to: {results_csv}")

## Visualize Degraded Images

Show examples of degraded images for each degradation type.


In [ ]:
def visualize_degraded_images(all_degraded_paths, degradation_type, image_name=None, max_levels=5):
    """
    Visualize degraded images for a specific degradation type.
    Shows original image first, then degraded variants.
    
    Args:
        all_degraded_paths: Dictionary mapping image_name to degraded_paths dict
        degradation_type: Type of degradation ('blur', 'exposure', 'jpeg')
        image_name: Specific image to show (if None, shows first image)
        max_levels: Maximum number of degradation levels to display (excluding original)
    """
    sns.set_style('whitegrid')
    image_names = list(all_degraded_paths.keys())
    
    for img_name in image_names:
        degraded_paths = all_degraded_paths[img_name]
        
        # Get original image
        original_path = degraded_paths.get('original')
        
        # Filter for this degradation type
        filtered_paths = {k: v for k, v in degraded_paths.items() if k.startswith(degradation_type)}
        
        # Sort by degradation level
        sorted_items = sorted(filtered_paths.items(), key=lambda x: x[0])
        selected_items = sorted_items[:max_levels]
        
        # Include original at the beginning
        display_items = [('Original', original_path)] if original_path else []
        display_items.extend(selected_items)
        
        n_images = len(display_items)
        fig, axes = plt.subplots(1, n_images, figsize=(4*n_images, 4))
        
        if n_images == 1:
            axes = [axes]
        
        for idx, (deg_name, img_path) in enumerate(display_items):
            img = cv2.imread(str(img_path))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(img_rgb)
            axes[idx].set_title(deg_name.replace('_', ' ').title(), fontsize=10)
            axes[idx].axis('off')
        
        plt.suptitle(f'{img_name} - {degradation_type.title()} Degradation', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Visualize degraded images for each degradation type
print("Visualizing degraded images...")
print("Showing first test image with different degradation levels\n")
for deg_type in ['blur', 'exposure', 'jpeg']:
    visualize_degraded_images(all_degraded_paths, deg_type, max_levels=5)


## Analysis: Dose-Response Curves

In [ ]:
def _prepare_degradation_data(df, degradation_type):
    """
    Prepare degradation data with normalized scores.
    Always creates both raw and normalized score columns.
    
    Args:
        df: DataFrame with columns: method, image_name, degradation_type, degradation_level, score
        degradation_type: Type of degradation ('blur', 'exposure', 'jpeg')
    
    Returns:
        Tuple of (df_filtered, xlabel)
    """
    df_filtered = df[df['degradation_type'] == degradation_type].copy()
    df_original = df[df['degradation_type'] == 'original'].copy()
    df_original['degradation_level'] = '0'
    df_filtered = pd.concat([df_original, df_filtered])

    # Create numeric level
    degradation_config = {
        'blur': ('sigma_', 'Blur Sigma'),
        'exposure': (None, 'Exposure Adjustment (stops)'),
        'jpeg': ('q_', 'JPEG Quality')
    }
    
    prefix, xlabel = degradation_config.get(degradation_type, (None, 'Degradation Level'))
    
    def parse_level(x):
        base_value = (100.0 if degradation_type == 'jpeg' else 0.0) if x == '0' else None
        if base_value is not None:
            return base_value
        
        exposure_replacements = {'plus_': '', 'minus_': '-'}
        replacements = exposure_replacements if degradation_type == 'exposure' else {prefix: ''} if prefix else {}
        
        value_str = x
        for old, new in replacements.items():
            value_str = value_str.replace(old, new)
        
        return float(value_str)
    
    df_filtered['deg_numeric'] = df_filtered['degradation_level'].apply(parse_level)

    # Always normalize scores per image using min-max scaling (range: 0 to 1)
    def normalize_group(x):
        min_val = x.min()
        max_val = x.max()
        range_val = max_val - min_val
        return (x - min_val) / range_val if range_val > 0 else x - min_val
    
    df_filtered['score_normalized'] = df_filtered.groupby(['method', 'image_name'])['score'].transform(normalize_group)
    
    return df_filtered, xlabel


def plot_dose_response(df, degradation_type):
    """
    Plot quality score vs degradation level (line plot).
    Shows both raw and normalized scores side by side.
    
    Args:
        df: DataFrame with columns: method, image_name, degradation_type, degradation_level, score
        degradation_type: Type of degradation ('blur', 'exposure', 'jpeg')
    """
    sns.set_style('whitegrid')
    df_filtered, xlabel = _prepare_degradation_data(df, degradation_type)
    
    # Average across images for both raw and normalized
    df_avg_raw = df_filtered.groupby(['method', 'deg_numeric'])['score'].mean().reset_index()
    df_avg_norm = df_filtered.groupby(['method', 'deg_numeric'])['score_normalized'].mean().reset_index()

    # Get unique methods and assign colors from tab20 palette
    methods = df_avg_raw['method'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(methods)))
    method_colors = dict(zip(methods, colors))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

    # Plot 1: Raw scores
    for method in methods:
        method_data = df_avg_raw[df_avg_raw['method'] == method]
        ax1.plot(method_data['deg_numeric'], method_data['score'], 
                marker='o', label=method, linewidth=2, markersize=6, color=method_colors[method])
    ax1.set_xlabel(xlabel, fontsize=12)
    ax1.set_ylabel('Quality Score (raw)', fontsize=12)
    ax1.set_title(f'{degradation_type.title()} Degradation: Raw Scores', fontsize=14, fontweight='bold')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax1.grid(True, alpha=0.3)
    if degradation_type == 'jpeg':
        ax1.invert_xaxis()

    # Plot 2: Normalized scores
    for method in methods:
        method_data = df_avg_norm[df_avg_norm['method'] == method]
        ax2.plot(method_data['deg_numeric'], method_data['score_normalized'], 
                marker='o', label=method, linewidth=2, markersize=6, color=method_colors[method])
    ax2.set_xlabel(xlabel, fontsize=12)
    ax2.set_ylabel('Normalized Quality Score (max=1.0 per image)', fontsize=12)
    ax2.set_title(f'{degradation_type.title()} Degradation: Normalized Scores', fontsize=14, fontweight='bold')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax2.grid(True, alpha=0.3)
    if degradation_type == 'jpeg':
        ax2.invert_xaxis()

    plt.tight_layout()
    plt.show()


def plot_dose_response_boxplot(df, degradation_type):
    """
    Plot box plot showing distribution of normalized scores across images.
    Uses fivecentplots. Each image is a data point.
    
    Args:
        df: DataFrame with columns: method, image_name, degradation_type, degradation_level, score
        degradation_type: Type of degradation ('blur', 'exposure', 'jpeg')
    """
    df_filtered, xlabel = _prepare_degradation_data(df, degradation_type)
    
    box_df = df_filtered.copy()
    box_df['deg_level_str'] = box_df['deg_numeric'].apply(lambda x: f'{x:.1f}')
    
    fcp.boxplot(
        box_df,
        y='score_normalized',
        groups='deg_level_str',
        wrap='method',
        xlabel=xlabel,
        ylabel='Normalized Quality Score (per image)',
        title=f'{degradation_type.title()} Degradation: Distribution Across Images',
        ax_size=[400, 400]
    )


# Plot for each degradation type
print("Plotting dose-response curves...")
for deg_type in ['blur', 'exposure', 'jpeg']:
    # Line plot (shows both raw and normalized)
    plot_dose_response(results_df, deg_type)
    
    # Box plot (normalized, showing distribution per image)
    # NOTE: Skipping boxplots to reduce number of plots
    # plot_dose_response_boxplot(results_df, deg_type)

In [ ]:
df_summary_1 = pd.read_csv("D:\sim-bench\outputs\pairwise_benchmark_test\pairwise_20251120_233139\methods_summary.csv")
df_summary_2 = pd.read_csv("D:\sim-bench\outputs\pairwise_benchmark_test\pairwise_20251121_004802\methods_summary.csv")
df_summary = pd.concat([df_summary_1, df_summary_2]).reset_index(drop=True)
df_summary.head(15)

## Summary Statistics

In [ ]:
df['majority_label'].nunique()

In [ ]:
df['majority_label'].value_counts()

In [ ]:
df =pd.read_csv(r"D:\Similar Images\automatic_triage_photo_series\photo_triage_pairs_embedding_labels.csv")
df[['series_id', 'compareID1', 'compareID2', 'compareFile1', 'compareFile2',
       'LEFT', 'RIGHT', 'Total', 'max_count', 'Agreement', 'MaxVote',
       'num_reviewers', 'majority_label']]

In [ ]:
# Compute score change for each method and degradation type
summary = []

for method in results_df['method'].unique():
    method_df = results_df[results_df['method'] == method]

    original_score = method_df[method_df['degradation_type'] == 'original']['score'].mean()

    for deg_type in ['blur', 'exposure', 'jpeg']:
        deg_scores = method_df[method_df['degradation_type'] == deg_type]['score']
        if len(deg_scores) > 0:
            worst_score = deg_scores.min()
            score_delta = original_score - worst_score

            summary.append({
                'method': method,
                'degradation': deg_type,
                'original_score': original_score,
                'worst_score': worst_score,
                'score_delta': score_delta,
                'sensitivity': 'High' if score_delta > 0.2 else ('Medium' if score_delta > 0.1 else 'Low')
            })

summary_df = pd.DataFrame(summary)

print("\n=== Method Sensitivity Summary ===")
print("\nScore Delta (Original - Worst):")
pivot = summary_df.pivot(index='method', columns='degradation', values='score_delta')
print(pivot.round(3))

print("\n=== Expected Behavior ===")
print("Sharpness method should be most sensitive to blur")
print("Exposure method should be most sensitive to exposure changes")
print("CLIP-Sharpness should detect blur degradation")
print("All methods should show decreasing scores with degradation")